# Data simulation



Vamos a generar los datos a través de una normal multivariada. En concreto:

$$
Z = (A, X) \sim \mathcal{N}(\mu, \Sigma)
$$

donde $A_l$ representan los atributos sensibles, $X_l$ los no sensibles y $\mu$ y $\Sigma$ son parámetros de la simulación a determinar. En nuestro caso, $\mu = \mathbf{0}$ y $\Sigma= \mathbf{I}$. Otros parámetros a determinar son las dimensiones de nuestros datos. 

En concreto nosotros vamos a tomar $30$ atributos no sensibles y $1$ o $2$ sensibles según que caso estemos estudiando: 

- Será $1$ en `Simulation_simple` y `Simulation_multalg`

- Mientras que en `Simulation_multsens` y `Simulation_general` usaremos $2$ atributos sensibles. 

Generaremos una matriz que tendrá tantas columnas como variables tengamos, y con tantas filas como observaciones queramos. En nuestro caso generaremos $1000$ observaciones.

Una vez hemos generada una matriz a partir de $(A_l, X_l)$ procederemos generando una variable más por medio de una relación lineal: 

$$
Y =  A \cdot \gamma +  X \cdot \beta = Z \cdot (\gamma , \beta)^\prime
$$

Donde $\beta$ y $\gamma$ son vectores a elegir. En nuestro caso consideraremos tres escenarios diferentes:

1. Alta correlación con los atributos sensibles $\hspace{0.1cm}\Rightarrow\hspace{0.1cm}$ $\gamma = \mathbf{1}$ ; $\beta=\mathbf{0}$.
2. Correlación moderada con los atributos sensibles $\hspace{0.1cm}\Rightarrow\hspace{0.1cm}$ $\gamma = \beta = \mathbf{1}$.
3. Baja correlación con los atributos sensibles $\hspace{0.1cm}\Rightarrow\hspace{0.1cm}$  $\gamma=\mathbf{0}$ ; $\beta=\mathbf{1}$.

Vamos a tratar con variables binarias por simplicidad. Así pues, las variables con subíndice $l$ que hemos generado representan las log-odds de las probabilidades de las variables reales. De tal modo, deshaciendo esta transformación por medio de la función logística obtendremos las probabilidades de las variables, es decir:

$$
p_J = logit(J_l) \quad \quad J \in \{Y, X, A\}
$$

Así pues, al elemento $x_{ij}$ de nuestro dataset le asignaremos un uno con probabilidad dada por la función logística aplicado al correspondiente elemento de la matriz generada a partir de una normal con los parámetros especificados. Una vez que hayamos convertido los valores simulados en unos y ceros, habremos obtenido ya nuestro dataset real: las variables reales $Y, X, A$ son aquellas generadas de esta forma.

## Requirements

In [1]:
import numpy as np
import pandas as pd
from aif360.datasets import BinaryLabelDataset

from PyFairnessAI.data import binary_data_simulation

In [2]:
n = 1000
p_sens = 2
p_no_sens = 10
random_state = 123

In [3]:
data_aif = binary_data_simulation(n=n, p_sens=p_sens, p_no_sens=p_no_sens, 
                                  mean=np.zeros(p_no_sens + p_sens), 
                                  cov=np.identity(p_no_sens + p_sens), 
                                  gamma=np.ones(p_sens), beta=np.zeros(p_no_sens), 
                                  random_state=random_state, 
                                  output_type='aif360')

In [4]:
data_pd = binary_data_simulation(n=n, p_sens=p_sens, p_no_sens=p_no_sens, 
                                 mean=np.zeros(p_no_sens + p_sens), 
                                 cov=np.identity(p_no_sens + p_sens), 
                                 gamma=np.ones(p_sens), beta=np.zeros(p_no_sens), 
                                 random_state=random_state, 
                                 output_type='pandas')

In [5]:
data_aif

               instance weights            features       ...           labels
                                protected attribute       ...                 
                                                A_1  A_2  ...  X_9 X_10       
instance names                                            ...                 
0                           1.0                 0.0  1.0  ...  1.0  0.0    1.0
1                           1.0                 1.0  1.0  ...  0.0  1.0    1.0
2                           1.0                 0.0  0.0  ...  1.0  1.0    1.0
3                           1.0                 0.0  1.0  ...  0.0  0.0    0.0
4                           1.0                 0.0  1.0  ...  1.0  1.0    1.0
...                         ...                 ...  ...  ...  ...  ...    ...
995                         1.0                 0.0  1.0  ...  1.0  1.0    0.0
996                         1.0                 0.0  0.0  ...  1.0  0.0    0.0
997                         1.0                 1.0 

In [6]:
data_pd

,Y,A_1,A_2,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10
0,1,0,1,1,0,0,1,0,0,1,0,1,0
1,1,1,1,1,0,1,1,1,1,1,0,0,1
2,1,0,0,1,0,1,0,1,0,0,0,1,1
3,0,0,1,1,0,0,0,1,1,1,1,0,0
4,1,0,1,0,0,1,0,0,1,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,1,1,1,0,0,1,0,1,0,1,1
996,0,0,0,0,0,0,0,0,0,0,0,1,0
997,1,1,1,0,1,0,1,0,0,1,1,1,1
998,1,1,0,0,0,1,0,0,0,0,0,0,1


In [7]:
data_aif.features.shape

(1000, 12)

In [8]:
data_aif.label_names # Response name

['Y']

In [9]:
data_aif.favorable_label # Y label of the favorable group

1.0

In [10]:
data_aif.unfavorable_label # Y label of the unfavorable group

0.0

In [11]:
data_aif.protected_attribute_names # Sensitive predictor names

['A_1', 'A_2']

In [12]:
data_aif.privileged_groups_sens # Label of the privileged group for each sensitive predictor

{'A_1': 1, 'A_2': 1}

In [13]:
data_aif.unprivileged_groups_sens # Label of the unprivileged group for each sensitive predictor

{'A_1': 0, 'A_2': 0}